# Lab 1: Quantum States and Measurement with NumPy

> **Target audience:** Physics students who completed Lab 0 and already know basic linear algebra, complex numbers, Dirac notation, and the Born rule.
> This lab comes **before Lecture 1** on quantum circuits and gates.

---

## Purpose

In this lab you will work entirely with **state vectors, matrices, and measurement rules** using NumPy.
The goal is to become comfortable with the computational side of single-qubit quantum mechanics **before** introducing gates and circuit diagrams.

You will build the mathematical objects yourself and use them on standard quantum-information tasks.


## What You Will Practice

| Part | Topic | Main ideas | Suggested time |
|---|---|---|---|
| 1 | State-vector toolkit | normalization, bras, inner products, projectors | 20 min |
| 2 | Measurement in different bases | Z, X, Y bases; probabilities; collapse | 25 min |
| 3 | Observables and expectation values | Pauli matrices, variances, physical interpretation | 20 min |
| 4 | Phase and time evolution | global vs relative phase, simple Hamiltonian evolution | 20 min |
| 5 | Optional extension | tensor products, Bell states, correlations | 15 min |

> **Core path:** Parts 1-4

> **Optional extension:** Part 5


## Lab Rules

1. Work in order.
2. Use **NumPy only** for the core tasks.
3. For each task, write the code, test it on at least one example, and explain the physics in one or two sentences.
4. When asked for interpretation, do not just report numbers. State what those numbers mean physically.
5. If you finish early, continue to the optional extension.

## Deliverables

By the end of the lab you should have:

- functions that manipulate quantum states and observables,
- numerical checks of standard single-qubit results,
- short written interpretations for each major part,
- at least one plot in Part 4.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(7)


def as_ket(v):
    arr = np.asarray(v, dtype=complex)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    return arr


def state_shape_ok(v, dim=2):
    return as_ket(v).shape == (dim, 1)


def norm(v):
    ket = as_ket(v)
    return float(np.sqrt(np.vdot(ket, ket).real))


def normalize(v):
    ket = as_ket(v)
    n = norm(ket)
    if np.isclose(n, 0.0):
        raise ValueError('Cannot normalize the zero vector.')
    return ket / n


def bra(v):
    return as_ket(v).conj().T


def inner(phi, psi):
    return np.vdot(as_ket(phi), as_ket(psi))


def projector(psi):
    ket = normalize(psi)
    return ket @ ket.conj().T


def is_valid_state(v, dim=2):
    ket = as_ket(v)
    return ket.shape == (dim, 1) and np.isclose(inner(ket, ket), 1.0)


def measurement_probabilities(psi, basis, labels=None):
    ket = normalize(psi)
    probs = np.array([abs(inner(b, ket)) ** 2 for b in basis], dtype=float)
    probs = probs / probs.sum()
    if labels is None:
        labels = [str(i) for i in range(len(basis))]
    return dict(zip(labels, probs))


def measure_state(psi, basis, labels=None, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    if labels is None:
        labels = [str(i) for i in range(len(basis))]
    probs_dict = measurement_probabilities(psi, basis, labels)
    probs = np.array(list(probs_dict.values()), dtype=float)
    idx = rng.choice(len(basis), p=probs)
    outcome_state = normalize(basis[idx])
    return {
        'label': labels[idx],
        'probabilities': probs_dict,
        'state': outcome_state,
    }


def expectation(psi, operator):
    ket = normalize(psi)
    value = (ket.conj().T @ operator @ ket).item()
    return np.real_if_close(value)


def variance(psi, operator):
    exp_a = expectation(psi, operator)
    exp_a2 = expectation(psi, operator @ operator)
    return float(np.real_if_close(exp_a2 - exp_a ** 2))


def unitary_from_hamiltonian(hamiltonian, t):
    eigenvalues, eigenvectors = np.linalg.eigh(hamiltonian)
    phases = np.exp(-1j * eigenvalues * t)
    return eigenvectors @ np.diag(phases) @ eigenvectors.conj().T


def computational_probabilities(psi):
    ket = normalize(psi).flatten()
    n = int(np.log2(len(ket)))
    labels = [format(i, f'0{n}b') for i in range(len(ket))]
    probs = np.abs(ket) ** 2
    return dict(zip(labels, probs))


ket0 = np.array([[1], [0]], dtype=complex)
ket1 = np.array([[0], [1]], dtype=complex)
ket_plus = normalize(ket0 + ket1)
ket_minus = normalize(ket0 - ket1)
ket_plus_i = normalize(ket0 + 1j * ket1)
ket_minus_i = normalize(ket0 - 1j * ket1)

basis_z = [ket0, ket1]
basis_x = [ket_plus, ket_minus]
basis_y = [ket_plus_i, ket_minus_i]
bases = {
    'Z': (basis_z, ['0', '1']),
    'X': (basis_x, ['+', '-']),
    'Y': (basis_y, ['+i', '-i']),
}

print('NumPy version:', np.__version__)
print('Shared toolkit loaded successfully.')



---

# Part 1 - Build a Minimal Quantum-State Toolkit

> **Goal:** Turn the abstract notation from Lab 0 into reusable computational tools.

In this part you will treat a qubit state as a complex column vector in $\mathbb{C}^2$ and implement the basic operations needed later in the lab.

A large part of beginner confusion in quantum information comes from switching too quickly between formulas, notation, and software without making the connections explicit. Here you will slow that process down. A ket such as $|\psi\rangle$ is not just a symbolic object: in computation it becomes an array of complex amplitudes, and every physically meaningful statement about the state must be recoverable from linear algebra. Normalization means total probability equals one, the bra is the conjugate transpose needed for complex inner products, and a projector is the matrix form of asking whether a state has a particular component along a direction in Hilbert space. By building these operations yourself, you are effectively creating a small numerical version of the postulates you already know from quantum mechanics.


### Task 1.1 - Validate and normalize states

Create tools that:

- check whether a candidate state has the correct shape,
- compute its norm,
- normalize it,
- verify numerically that $\langle \psi | \psi \rangle = 1$ after normalization.

Test your implementation on:

- a real vector proportional to $|0\rangle + |1\rangle$,
- a complex vector proportional to $|0\rangle + i|1\rangle$,
- one invalid input of your choice.

Write one sentence explaining why normalization is required in quantum mechanics.


In [ ]:
raw_real = np.array([1, 1], dtype=complex)
raw_complex = np.array([1, 1j], dtype=complex)
invalid_input = np.array([1, 2, 3], dtype=complex)

real_state = normalize(raw_real)
complex_state = normalize(raw_complex)

print('State shape checks:')
print('  raw_real ->', state_shape_ok(raw_real))
print('  raw_complex ->', state_shape_ok(raw_complex))
print('  invalid_input ->', state_shape_ok(invalid_input))

print()
print('Norms before normalization:')
print('  ||raw_real|| =', norm(raw_real))
print('  ||raw_complex|| =', norm(raw_complex))

print()
print('Normalized states:')
print('  normalize([1, 1]) =', real_state.flatten())
print('  normalize([1, i]) =', complex_state.flatten())
print('  <psi|psi> for real_state =', inner(real_state, real_state))
print('  <psi|psi> for complex_state =', inner(complex_state, complex_state))

assert state_shape_ok(raw_real)
assert state_shape_ok(raw_complex)
assert not state_shape_ok(invalid_input)
assert np.isclose(inner(real_state, real_state), 1.0)
assert np.isclose(inner(complex_state, complex_state), 1.0)
assert is_valid_state(real_state)
assert is_valid_state(complex_state)



### Task 1.2 - Bras, inner products, and projectors

Implement routines for:

- converting a ket $|\psi\rangle$ into a bra $\langle \psi |$,
- computing inner products $\langle \phi | \psi \rangle$,
- constructing projectors such as $|\psi\rangle\langle\psi|$.

Use your routines to verify the following claims:

1. $\langle 0|1\rangle = 0$.
2. $\langle +|-\rangle = 0$.
3. The projector onto $|0\rangle$ leaves $|0\rangle$ unchanged and annihilates $|1\rangle$.

Add a short explanation of what a projector means physically.


In [ ]:
P0 = projector(ket0)
ket0_after = P0 @ ket0
ket1_after = P0 @ ket1
inner_01 = inner(ket0, ket1)
inner_pm = inner(ket_plus, ket_minus)

print('Bra of |+> =', bra(ket_plus))
print('<0|1> =', inner_01)
print('<+|-> =', inner_pm)
print()
print('Projector |0><0| =')
print(P0)
print()
print('P0|0> =', ket0_after.flatten())
print('P0|1> =', ket1_after.flatten())

assert np.isclose(inner_01, 0.0)
assert np.isclose(inner_pm, 0.0)
assert np.allclose(ket0_after, ket0)
assert np.allclose(ket1_after, np.zeros_like(ket1))



### Checkpoint 1

Using only the tools from Part 1, compute the projector onto
$$
|+i\rangle = \frac{|0\rangle + i|1\rangle}{\sqrt{2}}.
$$

Then verify that applying this projector to $|+i\rangle$ returns the same state up to numerical precision.


In [ ]:
P_plus_i = projector(ket_plus_i)
recovered = P_plus_i @ ket_plus_i

print('Projector onto |+i> =')
print(P_plus_i)
print()
print('P_{+i}|+i> =', recovered.flatten())
print('Original |+i> =', ket_plus_i.flatten())

assert np.allclose(recovered, ket_plus_i)
assert np.allclose(P_plus_i @ P_plus_i, P_plus_i)



---

# Part 2 - Measurement in Different Bases

> **Goal:** See clearly that measurement outcomes depend on the basis, not only on the state vector entries written in one preferred form.

One of the most important conceptual shifts from classical information to quantum mechanics is that the same physical state can look simple or complicated depending on the basis in which it is described. Writing a qubit as a column vector in the computational basis can tempt you to think that those entries are the whole story, but they are only amplitudes relative to one chosen basis. Measurement is defined with respect to an orthonormal set of states, and changing that set changes the questions you are asking the system. This is why $|0\rangle$ is perfectly definite in the Z basis but gives a nontrivial distribution in the X basis, and why relative phase can be invisible in one basis yet observable in another. In this part, the Born rule becomes a practical tool rather than a sentence in a textbook.


### Task 2.1 - Define the standard single-qubit bases

Construct the basis vectors for:

- the computational basis: $\{|0\rangle, |1\rangle\}$,
- the X basis: $\{|+\rangle, |-\rangle\}$,
- the Y basis: $\{|+i\rangle, |-i\rangle\}$.

For each basis, verify numerically that:

- each basis vector is normalized,
- distinct basis vectors are orthogonal.

State what makes a set of vectors a valid measurement basis.


In [ ]:
for basis_name, (basis_vectors, labels) in bases.items():
    print(f'{basis_name} basis checks:')
    for label, vec in zip(labels, basis_vectors):
        print(f'  ||{label}||^2 =', inner(vec, vec))
        assert np.isclose(inner(vec, vec), 1.0)
    overlap = inner(basis_vectors[0], basis_vectors[1])
    print(f'  Overlap between basis vectors = {overlap}')
    assert np.isclose(overlap, 0.0)
    print()



### Task 2.2 - Compute measurement probabilities in an arbitrary basis

Write a routine that takes a normalized state $|\psi\rangle$ and an orthonormal basis $\{|b_0\rangle, |b_1\rangle\}$ and returns the probabilities
$$
P(b_k) = |\langle b_k|\psi\rangle|^2.
$$

Apply it to the following states:

1. $|0\rangle$
2. $|+\rangle$
3. $|+i\rangle$
4. $\sqrt{3/4}|0\rangle + \sqrt{1/4}|1\rangle$

For each state, compute probabilities in the Z, X, and Y bases.

Summarize your results in a table and comment on which basis makes each state look most "definite".


In [ ]:
states_to_test = {
    '|0>': ket0,
    '|+>': ket_plus,
    '|+i>': ket_plus_i,
    'sqrt(3/4)|0> + sqrt(1/4)|1>': normalize(np.array([np.sqrt(3/4), np.sqrt(1/4)], dtype=complex)),
}

summary = {}
for state_name, state in states_to_test.items():
    print(f'{state_name}:')
    summary[state_name] = {}
    for basis_name, (basis_vectors, labels) in bases.items():
        probs = measurement_probabilities(state, basis_vectors, labels)
        summary[state_name][basis_name] = probs
        max_label = max(probs, key=probs.get)
        print(f'  {basis_name} basis -> {probs} (most definite: {max_label})')
        assert np.isclose(sum(probs.values()), 1.0)
    print()

assert np.isclose(summary['|0>']['Z']['0'], 1.0)
assert np.isclose(summary['|+>']['X']['+'], 1.0)
assert np.isclose(summary['|+i>']['Y']['+i'], 1.0)



### Task 2.3 - Simulate projective measurement and collapse

Design a routine that performs an ideal projective measurement in a chosen basis.
It should:

- compute the possible outcomes and their probabilities,
- sample one outcome,
- return the post-measurement state.

Use this routine on the state $|+\rangle$ measured in the Z basis.
Repeat the experiment many times and estimate the frequencies.

Then test the collapse rule explicitly:

1. measure once in the Z basis,
2. immediately measure again in the same basis,
3. explain why the second result behaves differently from the first.


In [ ]:
trial_rng = np.random.default_rng(12345)
shots = 4000
outcomes = []
for _ in range(shots):
    result = measure_state(ket_plus, basis_z, ['0', '1'], rng=trial_rng)
    outcomes.append(result['label'])

freq_0 = outcomes.count('0') / shots
freq_1 = outcomes.count('1') / shots
print('Estimated frequencies for measuring |+> in Z basis:')
print({'0': freq_0, '1': freq_1})

repeat_rng = np.random.default_rng(2024)
matches = []
for _ in range(200):
    first = measure_state(ket_plus, basis_z, ['0', '1'], rng=repeat_rng)
    second = measure_state(first['state'], basis_z, ['0', '1'], rng=repeat_rng)
    matches.append(first['label'] == second['label'])

print('Second measurement matches first every time:', all(matches))

assert abs(freq_0 - 0.5) < 0.05
assert abs(freq_1 - 0.5) < 0.05
assert all(matches)



### Checkpoint 2

Find two different normalized states that give the **same probabilities in the Z basis** but **different probabilities in the X basis**.

Compute the probabilities explicitly and explain what physical feature distinguishes the two states.


In [ ]:
psi_a = ket_plus
psi_b = ket_plus_i

z_a = measurement_probabilities(psi_a, basis_z, ['0', '1'])
z_b = measurement_probabilities(psi_b, basis_z, ['0', '1'])
x_a = measurement_probabilities(psi_a, basis_x, ['+', '-'])
x_b = measurement_probabilities(psi_b, basis_x, ['+', '-'])

print('|psi_a> = |+>')
print('|psi_b> = |+i>')
print()
print('Z-basis probabilities:')
print('  psi_a ->', z_a)
print('  psi_b ->', z_b)
print()
print('X-basis probabilities:')
print('  psi_a ->', x_a)
print('  psi_b ->', x_b)

assert np.allclose(list(z_a.values()), list(z_b.values()))
assert not np.allclose(list(x_a.values()), list(x_b.values()))



---

# Part 3 - Observables, Expectation Values, and Uncertainty

> **Goal:** Connect qubit states to familiar quantum-mechanical observables.

In this part, treat the Pauli matrices as observables for a spin-1/2 system.

Up to this point, you have mainly described states and probabilities. Now the focus shifts to observables, which is where the qubit starts to look exactly like the spin-1/2 systems from standard quantum mechanics. The Pauli matrices are not introduced here as gates or circuit elements; they are Hermitian operators with real expectation values and well-defined measurement statistics. Computing $\langle \psi | A | \psi \rangle$ numerically forces you to connect algebraic expressions to physical meaning: an expectation value is not a mysterious average, but the statistical mean of repeated measurements on identically prepared systems. Variance then quantifies how sharp or uncertain the observable is in a given state. When a variance is zero, the state is an eigenstate of the observable, and when it is nonzero, the state carries intrinsic quantum uncertainty rather than simple ignorance.


### Task 3.1 - Expectation values of the Pauli observables

Define the matrices $\sigma_x$, $\sigma_y$, and $\sigma_z$.

Write a routine that computes the expectation value
$$
\langle A \rangle = \langle \psi | A | \psi \rangle.
$$

Evaluate $\langle \sigma_x \rangle$, $\langle \sigma_y \rangle$, and $\langle \sigma_z \rangle$ for:

- $|0\rangle$
- $|1\rangle$
- $|+\rangle$
- $|-\rangle$
- $|+i\rangle$

Interpret each result physically as a preferred measurement axis.


In [ ]:
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)

observable_states = {
    '|0>': ket0,
    '|1>': ket1,
    '|+>': ket_plus,
    '|->': ket_minus,
    '|+i>': ket_plus_i,
}

results = {}
for name, state in observable_states.items():
    values = {
        '<sigma_x>': float(np.real_if_close(expectation(state, sigma_x))),
        '<sigma_y>': float(np.real_if_close(expectation(state, sigma_y))),
        '<sigma_z>': float(np.real_if_close(expectation(state, sigma_z))),
    }
    results[name] = values
    print(name, '->', values)

assert np.isclose(results['|0>']['<sigma_z>'], 1.0)
assert np.isclose(results['|1>']['<sigma_z>'], -1.0)
assert np.isclose(results['|+>']['<sigma_x>'], 1.0)
assert np.isclose(results['|->']['<sigma_x>'], -1.0)
assert np.isclose(results['|+i>']['<sigma_y>'], 1.0)



### Task 3.2 - Variance and uncertainty

For an observable $A$, compute the variance
$$
\mathrm{Var}(A) = \langle A^2 \rangle - \langle A \rangle^2.
$$

Use your code to determine the variances of $\sigma_x$, $\sigma_y$, and $\sigma_z$ for the states $|0\rangle$, $|+\rangle$, and $|+i\rangle$.

Then answer:

- In which observable is each state perfectly sharp?
- In which observables is it uncertain?
- How does this relate to basis choice from Part 2?


In [ ]:
variance_states = {
    '|0>': ket0,
    '|+>': ket_plus,
    '|+i>': ket_plus_i,
}
observables = {'sigma_x': sigma_x, 'sigma_y': sigma_y, 'sigma_z': sigma_z}

variance_table = {}
for state_name, state in variance_states.items():
    variance_table[state_name] = {}
    print(state_name)
    for obs_name, obs in observables.items():
        var = variance(state, obs)
        variance_table[state_name][obs_name] = var
        print(f'  Var({obs_name}) = {var}')
    print()

assert np.isclose(variance_table['|0>']['sigma_z'], 0.0)
assert np.isclose(variance_table['|0>']['sigma_x'], 1.0)
assert np.isclose(variance_table['|+>']['sigma_x'], 0.0)
assert np.isclose(variance_table['|+i>']['sigma_y'], 0.0)



### Checkpoint 3

A student claims that the state $|+\rangle$ has "half spin-up and half spin-down in every possible measurement."

Use your numerical results to decide whether this claim is correct.
Support your answer with at least two explicit expectation values or probability calculations.


In [ ]:
probs_x_plus = measurement_probabilities(ket_plus, basis_x, ['+', '-'])
probs_z_plus = measurement_probabilities(ket_plus, basis_z, ['0', '1'])
probs_y_plus = measurement_probabilities(ket_plus, basis_y, ['+i', '-i'])

print('Probabilities for |+>:')
print('  X basis ->', probs_x_plus)
print('  Z basis ->', probs_z_plus)
print('  Y basis ->', probs_y_plus)
print()
print('Expectation values:')
print('  <sigma_x> =', expectation(ket_plus, sigma_x))
print('  <sigma_z> =', expectation(ket_plus, sigma_z))

assert np.isclose(probs_x_plus['+'], 1.0)
assert np.isclose(probs_z_plus['0'], 0.5)
assert np.isclose(probs_y_plus['+i'], 0.5)



---

# Part 4 - Phase and Time Evolution

> **Goal:** Show that phase is not just formal notation. Relative phase affects measurable predictions.

Students often accept complex amplitudes formally long before they feel why those phases matter physically. This part is meant to close that gap. A global phase multiplies the whole state by the same complex number and leaves every probability unchanged, so it has no observable effect. A relative phase, however, changes how amplitudes combine when the state is examined in another basis, and that is why interference exists at all. Time evolution provides the cleanest route to seeing this numerically. Under a Hamiltonian, components of the state acquire time-dependent phases, and even when populations in one basis stay fixed, probabilities in another basis can oscillate. In other words, phase is not decorative notation placed on top of real physics; it is part of the mechanism that determines measurable outcomes.


### Task 4.1 - Global phase versus relative phase

Compare the following three states:

$$
|\psi_1\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}, \qquad
|\psi_2\rangle = e^{i\pi/3}|\psi_1\rangle, \qquad
|\psi_3\rangle = \frac{|0\rangle + i|1\rangle}{\sqrt{2}}.
$$

Compute measurement probabilities for all three states in the Z, X, and Y bases.

Then explain:

- which two states are physically indistinguishable,
- which state differs by a relative phase,
- in which basis that difference becomes visible.


In [ ]:
psi_1 = ket_plus
psi_2 = np.exp(1j * np.pi / 3) * ket_plus
psi_3 = ket_plus_i
all_probs = {}

for name, state in {'psi_1': psi_1, 'psi_2': psi_2, 'psi_3': psi_3}.items():
    all_probs[name] = {}
    print(name)
    for basis_name, (basis_vectors, labels) in bases.items():
        probs = measurement_probabilities(state, basis_vectors, labels)
        all_probs[name][basis_name] = probs
        print(f'  {basis_name} basis -> {probs}')
    print()

for basis_name in bases:
    a = np.array(list(all_probs['psi_1'][basis_name].values()))
    b = np.array(list(all_probs['psi_2'][basis_name].values()))
    assert np.allclose(a, b)

assert np.allclose(list(all_probs['psi_1']['Z'].values()), list(all_probs['psi_3']['Z'].values()))
assert not np.allclose(list(all_probs['psi_1']['Y'].values()), list(all_probs['psi_3']['Y'].values()))



### Task 4.2 - Time evolution under a simple Hamiltonian

Consider the Hamiltonian
$$
H = \frac{\omega}{2}\sigma_z.
$$

Your task is to evolve an initial state in time using
$$
U(t) = e^{-iHt}.
$$

Suggested route:

- diagonalize the Hamiltonian numerically,
- construct the time-evolution operator from the eigenvalues and eigenvectors,
- evolve the initial state $|+\rangle$ over a range of times.

For each time, compute the probabilities of measuring in the X basis.

Create a plot of $P(+)$ and $P(-)$ as functions of time and explain the oscillation physically.


In [ ]:
omega = 1.0
H = 0.5 * omega * sigma_z
times = np.linspace(0.0, 2 * np.pi, 300)
prob_plus = []
prob_minus = []

for t in times:
    U_t = unitary_from_hamiltonian(H, t)
    psi_t = U_t @ ket_plus
    probs_x = measurement_probabilities(psi_t, basis_x, ['+', '-'])
    prob_plus.append(probs_x['+'])
    prob_minus.append(probs_x['-'])

prob_plus = np.array(prob_plus)
prob_minus = np.array(prob_minus)
expected_plus = np.cos(times / 2) ** 2
expected_minus = np.sin(times / 2) ** 2

plt.figure(figsize=(8, 4))
plt.plot(times, prob_plus, label='P(+ in X basis)', linewidth=2)
plt.plot(times, prob_minus, label='P(- in X basis)', linewidth=2)
plt.xlabel('time t')
plt.ylabel('probability')
plt.title('Evolution of |+> under H = (omega/2) sigma_z')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print('Maximum deviation from analytic result for P(+):', np.max(np.abs(prob_plus - expected_plus)))
print('Maximum deviation from analytic result for P(-):', np.max(np.abs(prob_minus - expected_minus)))

assert np.allclose(prob_plus, expected_plus)
assert np.allclose(prob_minus, expected_minus)



### Checkpoint 4

At what times in your simulation is the evolved state most similar to:

- $|+\rangle$,
- $|-\rangle$,
- $|+i\rangle$,
- $|-i\rangle$?

Use overlaps or measurement probabilities to justify your answer.


In [ ]:
omega = 1.0
H = 0.5 * omega * sigma_z
times_dense = np.linspace(0.0, 2 * np.pi, 401)
targets = {
    '|+>': ket_plus,
    '|->': ket_minus,
    '|+i>': ket_plus_i,
    '|-i>': ket_minus_i,
}

best_times = {}
for label, target in targets.items():
    overlaps = []
    for t in times_dense:
        psi_t = unitary_from_hamiltonian(H, t) @ ket_plus
        overlaps.append(abs(inner(target, psi_t)) ** 2)
    overlaps = np.array(overlaps)
    idx = int(np.argmax(overlaps))
    best_times[label] = times_dense[idx]
    print(f'{label} best match at t = {times_dense[idx]:.4f} with overlap {overlaps[idx]:.4f}')

assert np.isclose(best_times['|+>'], 0.0) or np.isclose(best_times['|+>'], 2 * np.pi)
assert np.isclose(best_times['|+i>'], np.pi / 2, atol=0.02)
assert np.isclose(best_times['|->'], np.pi, atol=0.02)
assert np.isclose(best_times['|-i>'], 3 * np.pi / 2, atol=0.02)



---

# Part 5 - Optional Extension: Two-Qubit States Without Circuits

> **Optional:** Attempt this only if you have completed the core path.

This part previews multi-qubit quantum information using only tensor products and state vectors.

The purpose of this extension is to show that multi-qubit quantum information does not begin with circuit notation. It begins with the structure of composite Hilbert spaces. When two qubits are combined, the dimension of the state space multiplies rather than adds, and the tensor product is the mathematical rule that builds the joint system from the subsystems. Product states are the straightforward combinations you would expect from independent systems, but entangled states are different: the full state is well defined even though it cannot be factored into a state of qubit A times a state of qubit B. That distinction is central to quantum information. By constructing both product states and Bell states directly as vectors, you can study correlations before any formal introduction of quantum gates or circuit diagrams.


### Task 5.1 - Build product states with tensor products

Using `np.kron`, construct the states:

- $|00\rangle$
- $|01\rangle$
- $|10\rangle$
- $|11\rangle$
- $|+0\rangle$
- $|++\rangle$

Verify normalization and compute the probabilities of all computational-basis outcomes.

Explain how the dimension changes when going from one qubit to two.


In [ ]:
ket00 = np.kron(ket0, ket0).reshape(-1, 1)
ket01 = np.kron(ket0, ket1).reshape(-1, 1)
ket10 = np.kron(ket1, ket0).reshape(-1, 1)
ket11 = np.kron(ket1, ket1).reshape(-1, 1)
ket_plus0 = np.kron(ket_plus, ket0).reshape(-1, 1)
ket_plusplus = np.kron(ket_plus, ket_plus).reshape(-1, 1)

two_qubit_states = {
    '|00>': ket00,
    '|01>': ket01,
    '|10>': ket10,
    '|11>': ket11,
    '|+0>': ket_plus0,
    '|++>': ket_plusplus,
}

for label, state in two_qubit_states.items():
    probs = computational_probabilities(state)
    print(label, 'shape =', state.shape, 'probabilities =', probs)
    assert state.shape == (4, 1)
    assert np.isclose(sum(probs.values()), 1.0)

print()
print('Dimension change: one qubit -> 2 amplitudes, two qubits -> 4 amplitudes.')



### Task 5.2 - Compare a product state and a Bell state

Construct the Bell state
$$
|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}.
$$

Compare it with the product state $|++\rangle$.

For both states:

- compute computational-basis probabilities,
- compute the probability that the two qubits give the same measurement result in the Z basis,
- write two sentences describing the difference between "superposition" and "entanglement".


In [ ]:
phi_plus = normalize(ket00 + ket11)
plusplus = ket_plusplus

phi_probs = computational_probabilities(phi_plus)
plusplus_probs = computational_probabilities(plusplus)

same_phi = phi_probs['00'] + phi_probs['11']
same_plusplus = plusplus_probs['00'] + plusplus_probs['11']

print('|Phi+> probabilities =', phi_probs)
print('|++> probabilities =', plusplus_probs)
print()
print('Probability of same Z-basis outcomes:')
print('  |Phi+> ->', same_phi)
print('  |++> ->', same_plusplus)

assert np.isclose(phi_probs['00'], 0.5)
assert np.isclose(phi_probs['11'], 0.5)
assert np.isclose(same_phi, 1.0)
assert np.isclose(same_plusplus, 0.5)



---

# Fast Finishers

> **Bonus:** These tasks are for students who complete the core lab early.

The goal here is not to introduce a completely new topic, but to deepen the same ideas from the main lab. Each bonus task uses the same NumPy-based language of vectors, matrices, probabilities, and simple time evolution. Together they add a more advanced layer that should comfortably fill the remainder of a 90-minute lab for stronger students.



### Bonus 1 - Density Matrices for Pure States

A pure-state density matrix is defined by
$$
\rho = |\psi\rangle\langle\psi|.
$$

For a pure state, the density matrix is Hermitian, has trace 1, and satisfies $\rho^2 = \rho$. This gives you a second representation of the same physical state: instead of storing amplitudes as a ket, you store all measurement information in a matrix.

Tasks:

- implement a function `density_matrix(psi)`,
- compute `rho_plus_i` for $|+i\rangle$,
- verify numerically that `trace(rho_plus_i) = 1`,
- verify numerically that `rho_plus_i @ rho_plus_i = rho_plus_i`.


In [ ]:
def density_matrix(psi):
    ket = normalize(psi)
    return ket @ ket.conj().T

rho_plus_i = density_matrix(ket_plus_i)
trace_rho_plus_i = np.trace(rho_plus_i)
purity_rho_plus_i = np.trace(rho_plus_i @ rho_plus_i)

print('rho_plus_i =')
print(rho_plus_i)
print()
print('trace(rho_plus_i) =', trace_rho_plus_i)
print('trace(rho_plus_i^2) =', purity_rho_plus_i)

assert rho_plus_i.shape == (2, 2)
assert np.allclose(rho_plus_i, rho_plus_i.conj().T)
assert np.isclose(trace_rho_plus_i, 1.0)
assert np.allclose(rho_plus_i @ rho_plus_i, rho_plus_i)
assert np.isclose(purity_rho_plus_i, 1.0)


### Bonus 2 - Expectation Values from the Trace Formula

Once you have a density matrix, expectation values can be computed using
$$
\langle A \rangle = \mathrm{Tr}(\rho A).
$$

For pure states this should agree exactly with the ket-based formula $\langle\psi|A|\psi\rangle$. This bonus task is useful because it connects the bra-ket language to the matrix-based formalism that will later become essential for mixed states and subsystems.

Tasks:

- implement a function `trace_expectation(rho, operator)`,
- compute expectation values of $\sigma_x$, $\sigma_y$, and $\sigma_z$ for `rho_plus_i`,
- compare those values with the earlier ket-based results.


In [ ]:
def trace_expectation(rho, operator):
    value = np.trace(np.asarray(rho, dtype=complex) @ np.asarray(operator, dtype=complex))
    return np.real_if_close(value)

trace_exp_x_plus_i = trace_expectation(rho_plus_i, sigma_x)
trace_exp_y_plus_i = trace_expectation(rho_plus_i, sigma_y)
trace_exp_z_plus_i = trace_expectation(rho_plus_i, sigma_z)

print('Trace-based expectation values for rho_plus_i:')
print('  <sigma_x> =', trace_exp_x_plus_i)
print('  <sigma_y> =', trace_exp_y_plus_i)
print('  <sigma_z> =', trace_exp_z_plus_i)

assert np.isclose(trace_exp_x_plus_i, expectation(ket_plus_i, sigma_x))
assert np.isclose(trace_exp_y_plus_i, expectation(ket_plus_i, sigma_y))
assert np.isclose(trace_exp_z_plus_i, expectation(ket_plus_i, sigma_z))
assert np.isclose(trace_exp_y_plus_i, 1.0)



### Bonus 3 - Rabi Oscillations Under $H = \frac{\omega}{2}\sigma_x$

In Part 4 you evolved a state under $\sigma_z$ and looked at probabilities in the X basis. Here you will rotate the problem: evolve the initial state $|0\rangle$ under the Hamiltonian
$$
H_x = \frac{\omega}{2}\sigma_x
$$
and measure in the Z basis.

Because $\sigma_x$ couples $|0\rangle$ and $|1\rangle$, you should see oscillations between the two computational-basis outcomes. This is the same linear-algebra machinery as before, but with a different observable and a different measurement basis.

Tasks:

- reuse `unitary_from_hamiltonian`,
- evolve $|0\rangle$ over a range of times,
- compute `prob_0_x` and `prob_1_x`,
- verify that they match $\cos^2(t/2)$ and $\sin^2(t/2)$,
- plot both curves.


In [ ]:
omega_x = 1.0
Hx = 0.5 * omega_x * sigma_x
times_x = np.linspace(0.0, 2 * np.pi, 300)
prob_0_x = []
prob_1_x = []

for t in times_x:
    psi_t = unitary_from_hamiltonian(Hx, t) @ ket0
    probs_z = measurement_probabilities(psi_t, basis_z, ['0', '1'])
    prob_0_x.append(probs_z['0'])
    prob_1_x.append(probs_z['1'])

prob_0_x = np.array(prob_0_x)
prob_1_x = np.array(prob_1_x)
expected_0_x = np.cos(times_x / 2) ** 2
expected_1_x = np.sin(times_x / 2) ** 2

plt.figure(figsize=(8, 4))
plt.plot(times_x, prob_0_x, label='P(0 in Z basis)', linewidth=2)
plt.plot(times_x, prob_1_x, label='P(1 in Z basis)', linewidth=2)
plt.xlabel('time t')
plt.ylabel('probability')
plt.title('Evolution of |0> under H = (omega/2) sigma_x')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print('Maximum deviation for P(0):', np.max(np.abs(prob_0_x - expected_0_x)))
print('Maximum deviation for P(1):', np.max(np.abs(prob_1_x - expected_1_x)))

assert np.allclose(prob_0_x + prob_1_x, 1.0)
assert np.allclose(prob_0_x, expected_0_x)
assert np.allclose(prob_1_x, expected_1_x)
assert np.isclose(prob_0_x[0], 1.0)



---

# Wrap-Up

## What you should understand after this lab

- A qubit state is a normalized complex vector.
- Measurement probabilities depend on the chosen basis.
- Projective measurement changes the state.
- Pauli matrices can be treated as observables for spin-1/2 systems.
- Relative phase affects measurable predictions.
- Time evolution can be simulated numerically with linear algebra.

## Reflection questions

1. Which part of this lab felt most like standard quantum mechanics?
2. Which part felt most like quantum information?
3. What became clearer when you computed everything numerically rather than only symbolically?
4. What do you expect a "quantum gate" to be, based on the operations you used here?

## Forward link to Lecture 1

In Lecture 1, these same objects will be reorganized into the language of:

- quantum gates as unitary operators,
- circuit diagrams,
- measurement as part of a computational workflow,
- multi-qubit quantum processes.
